In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [3]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [6]:
from starter import index
from rag_helper import RAGBase


class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            messages = [
                {"role": "system", "content": self.instructions},
                {"role": "user", "content": prompt},
            ]

            response = self.llm_client.chat.completions.create(
                model=self.model,
                messages=messages,
            )

            # wrap so it looks like the `responses` API object rag_helper.py expects
            class Wrapped:
                output_text = response.choices[0].message.content

            wrapped = Wrapped()

            # stash raw response and usage on the span object for Q2 later
            span.set_attribute("_has_usage", response.usage is not None)

            return wrapped


traced_rag = RAGTraced(
    index=index,
    llm_client=openai_client,
    model="gemini-2.5-flash"
)

In [7]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = traced_rag.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xfbc133c2568e9fe3128d625561ba9cbd",
        "span_id": "0x8d83640e0807a6c7",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x3139c32f963e79e8",
    "start_time": "2026-07-19T13:43:52.371423Z",
    "end_time": "2026-07-19T13:43:52.375641Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "6c479ac0-ed14-44fe-b2e6-06763ad22030",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xfbc133c2568e9fe3128d625561ba9cbd",
        "span_id": "0x8551d47498ac07ae",
        "trace_state": "[]"
    },
    "kind": "SpanKind

In [8]:
class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            messages = [
                {"role": "system", "content": self.instructions},
                {"role": "user", "content": prompt},
            ]

            response = self.llm_client.chat.completions.create(
                model=self.model,
                messages=messages,
            )

            usage = response.usage
            span.set_attribute("input_tokens", usage.prompt_tokens)
            span.set_attribute("output_tokens", usage.completion_tokens)

            class Wrapped:
                output_text = response.choices[0].message.content

            return Wrapped()

traced_rag = RAGTraced(
    index=index,
    llm_client=openai_client,
    model="gemini-2.5-flash"
)

query = "How does the agentic loop keep calling the model until it stops?"
answer = traced_rag.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x02a3c05e82e84fbbabe371c1ca92f5de",
        "span_id": "0x0ee549750c5dc09c",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xd2691255426eef97",
    "start_time": "2026-07-19T13:51:07.942223Z",
    "end_time": "2026-07-19T13:51:07.944847Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "6c479ac0-ed14-44fe-b2e6-06763ad22030",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x02a3c05e82e84fbbabe371c1ca92f5de",
        "span_id": "0x8000bae734c23e6a",
        "trace_state": "[]"
    },
    "kind": "SpanKind

Restart kernel from here since need to swap processor

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [2]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [3]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter("traces.db")))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [4]:
from starter import index
from rag_helper import RAGBase


class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            messages = [
                {"role": "system", "content": self.instructions},
                {"role": "user", "content": prompt},
            ]
            response = self.llm_client.chat.completions.create(
                model=self.model,
                messages=messages,
            )
            usage = response.usage
            span.set_attribute("input_tokens", usage.prompt_tokens)
            span.set_attribute("output_tokens", usage.completion_tokens)

            class Wrapped:
                output_text = response.choices[0].message.content

            return Wrapped()


traced_rag = RAGTraced(index=index, llm_client=openai_client, model="gemini-2.5-flash")

In [5]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = traced_rag.rag(query)
print(answer)

The agentic loop keeps calling the model until the model returns a response *without any function calls*.

Here's how it works:
1.  The loop sends the current message history to the model.
2.  The model processes the input and returns an output.
3.  The code checks the model's output:
    *   If the output contains a `function_call`, the agent runs the tool, appends the tool's output to the message history, and continues the loop (i.e., makes another call to the model).
    *   If the output contains only a `message` (i.e., no function calls), it means the model has a final answer and doesn't need to use any more tools. In this case, the `has_function_calls` flag remains `False`, and the loop breaks.

Essentially, the model itself dictates when it's "done" by choosing not to request any more tool executions.


In [6]:
import pandas as pd

df = pd.read_sql("SELECT * FROM spans", sqlite3.connect("traces.db"))
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784470029602011402,1784470029607867380,NaN,NaN,None
1,llm,1784470029630939485,1784470033141010699,7933.0,205.0,None
2,rag,1784470029601922156,1784470033156619797,NaN,NaN,None


In [7]:
df = pd.read_sql("SELECT * FROM spans", sqlite3.connect("traces.db"))
df["duration_ns"] = df["end_time"] - df["start_time"]

totals = (
    df[df["name"] != "rag"]
    .groupby("name")["duration_ns"]
    .sum()
    .sort_values(ascending=False)
)
print(totals)

name
llm       3510071214
search       5855978
Name: duration_ns, dtype: int64


In [8]:
query = "How does the agentic loop keep calling the model until it stops?"

for _ in range(3):
    traced_rag.rag(query)

In [9]:
df = pd.read_sql("SELECT * FROM spans", sqlite3.connect("traces.db"))
llm_tokens = df[df["name"] == "llm"]["input_tokens"]
print(llm_tokens)
print(llm_tokens.min(), llm_tokens.max())

1     7933.0
4     7933.0
7     7933.0
10    7933.0
Name: input_tokens, dtype: float64
7933.0 7933.0
